# 基于MindSpore 2.7.0与MindNLP 0.5.1的T5邮件摘要微调与推理

目标：在原始notebook的前提下，生成适配MindSpore 2.7.0与MindNLP 0.5.1的新版本，完成T5模型的微调与推理流程。

## 环境准备

In [1]:
!pip install mindnlp==0.5.1

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 20.2 MB/s  0:00:00


## 版本检查

In [1]:
import mindspore as ms
import mindnlp
print('MindSpore:', ms.__version__)
print('MindNLP:', getattr(mindnlp, '__version__', 'unknown'))


/root/.conda/envs/mindspore/lib/python3.11/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/root/.conda/envs/mindspore/lib/python3.11/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/root/.conda/envs/mindspore/lib/python3.11/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/root/.conda/envs/mindspore/lib/python3.11/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/root/.conda/envs/mindspore/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarn

MindSpore: 2.7.0
MindNLP: 0.5.0rc2


![alt text](image.png)
我下载mindnlp==0.5.1，但是显示的是0.5.0rc2，官网下面也有人是这样的，如图所示

## 数据集加载
通过 MindNLP 的 load_dataset 接口加载了经典的文本摘要数据集 CNN/DailyMail（版本 3.0.0），并自动将其划分为训练集、验证集和测试集三个部分，最后打印各子集的样本数量以验证数据下载及加载过程是否完整无误。

In [2]:
from mindnlp.dataset import load_dataset
ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=['train', 'validation', 'test'])
train = ds['train']
val = ds['validation']
test = ds['test']
print('train:', len(train), 'validation:', len(val), 'test:', len(test))

train: 287113 validation: 13368 test: 11490


## 分词器与编码设置
加载了 t5-base 模型的预训练分词器，并设置了 T5 模型所需的特定前缀 summarize: ，同时定义了输入文本的最大长度（512）和目标摘要的最大长度（64），最后获取了填充符号（pad_token）的 ID，为后续将文本转换为模型可读的数字序列做准备。

In [3]:
from mindnlp.transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('t5-base')
prefix = 'summarize: '
max_input_length = 512
max_target_length = 64
pad_id = tokenizer.pad_token_id
print('pad_token_id:', pad_id)

pad_token_id: 0


## 生成式数据集
定义了一个自定义的 Python 类 SummarizationDataset 来处理原始数据，在 __getitem__ 方法中它会对文章加上前缀并进行分词，同时对目标摘要进行编码，最关键的是它将标签中的填充位置（pad_token_id）替换为 -100，这样在计算损失函数时模型就会自动忽略这些填充部分，从而保证训练的准确性。

In [4]:
import numpy as np
class SummarizationDataset:
    def __init__(self, data):
        self.data = data
    def __getitem__(self, idx):
        art, summ, _ = self.data[idx]
        enc = tokenizer(prefix + art, max_length=max_input_length, padding='max_length', truncation=True)
        dec = tokenizer(summ, max_length=max_target_length, padding='max_length', truncation=True)
        input_ids = np.array(enc.input_ids, dtype=np.int32)
        attention_mask = np.array(enc.attention_mask, dtype=np.int32)
        labels = np.array(dec.input_ids, dtype=np.int32)
        labels = np.where(labels == pad_id, -100, labels).astype(np.int32)
        return input_ids, attention_mask, labels
    def __len__(self):
        return len(self.data)

## 构建MindSpore数据管线
将前面定义的 Python 数据集类转换为 MindSpore 的高性能数据管道，使用 GeneratorDataset 接入数据后，通过 TypeCast 操作将所有输入数据转换为 MindSpore 计算所需的 int32 类型，并设置批次大小为 8，同时丢弃最后不足一个批次的数据，以确保训练时的 tensor 维度固定。

In [5]:
import mindspore.dataset as ds
from mindspore.dataset import transforms
from mindspore import dtype as mstype
train_ds = ds.GeneratorDataset(SummarizationDataset(train), column_names=['input_ids','attention_mask','labels'], shuffle=True)
val_ds = ds.GeneratorDataset(SummarizationDataset(val), column_names=['input_ids','attention_mask','labels'], shuffle=False)
for c in ['input_ids','attention_mask','labels']:
    train_ds = train_ds.map(operations=transforms.TypeCast(mstype.int32), input_columns=c)
    val_ds = val_ds.map(operations=transforms.TypeCast(mstype.int32), input_columns=c)
batch_size = 8
train_ds = train_ds.batch(batch_size, drop_remainder=True)
val_ds = val_ds.batch(batch_size, drop_remainder=True)
print('train batches:', train_ds.get_dataset_size(), 'val batches:', val_ds.get_dataset_size())

train batches: 35889 val batches: 1671


## 构建模型与训练网络
微调的核心逻辑部分，首先通过一段兼容性代码处理了 mindtorch.autograd.profiler 以防止环境报错，接着加载预训练的 T5ForConditionalGeneration 模型并将其设置为训练模式，随后定义了 AdamW 优化器，并编写了 train_step 函数，该函数封装了单步训练的完整流程，包括前向计算、获取损失值、反向传播求导、梯度裁剪以及参数更新。

In [ ]:
import contextlib
import mindtorch.autograd as autograd
import mindtorch as mt
from mindnlp.transformers import T5ForConditionalGeneration

try:
    import mindtorch.autograd.profiler as _prof
    autograd.profiler = _prof
except Exception:
    class _DummyProfiler:
        @staticmethod
        @contextlib.contextmanager
        def record_function(name):
            yield
    autograd.profiler = _DummyProfiler()


model = T5ForConditionalGeneration.from_pretrained("t5-base")

model.train()


lr = 1e-4
weight_decay = 0.01
optimizer = mt.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)


clip_norm = 1.0

def to_long(x):
    if hasattr(x, "asnumpy"):
        x = x.asnumpy()
    return mt.tensor(x, dtype=mt.long)

def train_step(input_ids, attention_mask, labels):
    input_ids = to_long(input_ids)
    attention_mask = to_long(attention_mask)
    labels = to_long(labels)

    optimizer.zero_grad()
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = outputs.loss

    loss.backward()
    mt.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
    optimizer.step()

    return float(loss.item())


## 训练
执行实际的训练过程，设置了只训练 1 个 Epoch 并且仅运行 200 个 Step 用于快速演示代码的可行性，循环中使用 tqdm 进度条实时显示训练进度，通过迭代 MindSpore 的数据管道获取数据并传入 train_step 函数进行权重更新，每隔 20 步打印一次当前的 Loss 值以监控收敛情况。

In [7]:
from tqdm import tqdm

num_epochs = 1
warm_steps = 200

for epoch in range(num_epochs):
    step = 0
    with tqdm(total=warm_steps, desc=f"Epoch {epoch+1}/{num_epochs}", unit="step") as pbar:
        for input_ids, attention_mask, labels in train_ds.create_tuple_iterator():
            loss = train_step(input_ids, attention_mask, labels)  
            step += 1

            if step % 20 == 0:
                tqdm.write(f"step={step}, loss={loss:.4f}")

            pbar.update(1)
            if step >= warm_steps:
                break


Epoch 1/1:  10%|█         | 20/200 [13:34<2:01:02, 40.35s/step]

step=20, loss=1.9781


Epoch 1/1:  20%|██        | 40/200 [27:06<1:48:26, 40.67s/step]

step=40, loss=2.1115


Epoch 1/1:  30%|███       | 60/200 [40:33<1:33:32, 40.09s/step]

...

step=160, loss=2.3649


Epoch 1/1:  90%|█████████ | 180/200 [1:59:24<13:00, 39.02s/step]

step=180, loss=2.1277


Epoch 1/1: 100%|██████████| 200/200 [2:12:27<00:00, 39.74s/step]

step=200, loss=2.1511


## 推理
验证模型的实际生成效果，首先定义了一段关于 AI 医疗的长文本并加上摘要前缀进行编码，然后将其转换为 MindSpore Tensor 输入模型，调用 model.generate 方法生成摘要序列（设置了最大长度和温度参数），最后通过分词器解码生成的 ID 序列，打印出模型生成的文本摘要。

In [8]:
text = """Artificial Intelligence (AI) has rapidly evolved and become an integral part of various industries, 
particularly healthcare. AI's ability to analyze large volumes of data and identify patterns that humans might miss 
has revolutionized many aspects of medical care, 
from diagnostics to treatment planning and patient monitoring.
One of the most significant contributions of AI in healthcare is in the field of diagnostics. 
Machine learning algorithms, a subset of AI, have been trained to interpret medical imaging data, such as X-rays, MRIs, and CT scans.
These algorithms can often detect abnormalities, such as tumors or fractures, with higher accuracy and speed than human radiologists. 
This not only improves the speed of diagnosis but also reduces the potential for human error, leading to better patient outcomes.
AI has also shown promise in personalized medicine, where treatments are tailored to an individual's genetic makeup and health history.
By analyzing patient data, including genetic information, 
AI systems can recommend specific treatment plans that are more likely to be effective. 
This can greatly enhance the precision of treatments, reduce side effects, and improve overall patient care.."""
enc = tokenizer(prefix + text, max_length=max_input_length, padding='max_length', truncation=True)
inp = ms.Tensor([enc.input_ids], dtype=ms.int32)
gen_ids = model.generate(inp, max_length=60,top_k=0,temperature=0.7)
summ = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
print(summ)

The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


AI has become an integral part of many industries, including healthcare . the ability to analyze large volumes of data has revolutionized many aspects of care .


## 保存模型与分词器
将微调后的模型权重以及对应的分词器配置文件保存到本地目录 ./t5_email_summarization_ms270_mn051 中，这样做是为了持久化训练成果，方便后续重新加载模型进行推理服务或继续训练，而无需从头开始。

In [9]:
save_dir = './t5_email_summarization_ms270_mn051'
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print('saved to', save_dir)

saved to ./t5_email_summarization_ms270_mn051
